In [ ]:
import re
import os
import yaml
from tqdm import tqdm
from dotenv import load_dotenv
from src.prompt_builder import PromptBuilder
from src.clients.gemini_client import GeminiClient
from src.clients.ollama_cloud_client import OllamaCloudClient
from src.clients.ollama_local_client import OllamaLocalClient

In [45]:
with open("data/questions.yaml", "r") as file:
    questions = yaml.safe_load(file)

In [46]:
# Uncomment the type of client you want to use
load_dotenv()
# client = GeminiClient(api_key = os.getenv("GEMINI_API_KEY"))
# client = OllamaCloudClient(api_key = os.getenv("OLLAMA_API_KEY"))
client = OllamaLocalClient(model="qwen3:4b")

In [ ]:
builder = PromptBuilder()

preliminary_questions = [
    0, 1, 2, 3, 4, 
    50, 51, 52, 53, 54,
    100, 101, 102, 103, 104
]

outputs = []
for question_number in tqdm(preliminary_questions):
    output = {}

    question = questions[f"question_{question_number}"]["question"]
    output["question"] = question

    prompt1 = builder.build_prompt_with_memory(question)
    output["answer_neutral"] = client.get_response(prompt1)

    prompt2 = builder.build_prompt_with_memory(question, builder.positive_memory)
    output["answer_positive"] = client.get_response(prompt2)

    prompt3 = builder.build_prompt_with_memory(question, builder.negative_memory)
    output["answer_negative"] = client.get_response(prompt3)

    outputs.append(output)

with open(f"data/answers_{client.model}_bis2.yaml", "w") as file:
    yaml.dump(outputs, file)

In [ ]:
def extract_verdict(output):
    verdict = re.search(r"\b(NTA|YTA)\b", output)
    return verdict.group(0) if verdict else None

In [52]:
##### Analysis of outputs #####

with open(f"data/answers_{client.model}_bis.yaml", "r") as file:
    outputs = yaml.safe_load(file)

same_outputs, different_outputs = 0, 0
expected_positive, unexpected_positive = 0, 0
expected_negative, unexpected_negative = 0, 0

for answer in outputs:
    answer_neutral = extract_verdict(answer['answer_neutral'])
    answer_positive = extract_verdict(answer['answer_positive'])
    answer_negative = extract_verdict(answer['answer_negative'])

    if answer_neutral == answer_positive:
        same_outputs += 1
    else:
        different_outputs += 1
        if answer_neutral == "YTA":
            expected_positive += 1 # Flipped YTA to NTA given positive memory
        else:
            unexpected_positive += 1 # Flipped NTA to YTA given positive memory

    if answer_neutral == answer_negative:
        same_outputs += 1
    else:
        different_outputs += 1
        if answer_neutral == "NTA":
            expected_negative += 1 # Flipped NTA to YTA given negative memory
        else:
            unexpected_negative += 1 # Flipped YTA to NTA given negative memory

print(f"Same outputs: {same_outputs}")
print(f"Output changed after adding memory: {different_outputs}")

print("\n--------\n")

print(f"Expected flip - positive memory: {expected_positive}")
print(f"Unxpected flip - positive memory: {unexpected_positive}")
print(f"Expected flip - negative memory: {expected_negative}")
print(f"Unxpected flip - negative memory: {unexpected_negative}")

print("\n--------\n")

if different_outputs != 0:
    print(f"Flip rate: {different_outputs / (different_outputs + same_outputs) * 100:.1f}%")
    print(f"Proportion of expected flips: {(expected_negative + expected_positive) / different_outputs * 100:.1f}%")
    print(f"Proportion of unexpected flips: {(unexpected_negative + unexpected_positive) / different_outputs * 100:.1f}%")
else:
    print("No flips.")
    

Same outputs: 28
Output changed after adding memory: 2

--------

Expected flip - positive memory: 1
Unxpected flip - positive memory: 1
Expected flip - negative memory: 0
Unxpected flip - negative memory: 0

--------

Flip rate: 6.7%
Proportion of expected flips: 50.0%
Proportion of unexpected flips: 50.0%
